# Retries, Timeouts, Rate Limits & Circuit Breakers

## What you'll practice

Remote systems fail transiently: 429s, 503s, timeouts, dropped connections. Production clients retry *only* what's retryable, back off exponentially, honour
`Retry-After`, cap their own request rate, and stop hammering a dead dependency (circuit breaker). Every time-related piece takes an injectable `sleep`/`clock`
so tests run instantly.

> Function and class names in **Implement** are what the tests look for. Run all tests for this notebook with `pytest tests/apis/test_07_04_retries_rate_limits.py -v` from the lab root. Untouched exercises show as *skipped*.
>
> Solutions: `solutions/07_apis/04_retries_rate_limits.ipynb`. Don't open it until you've tried.

### Setup — run this first (sample data and imports, not a solution)

In [ ]:
import sys
from pathlib import Path
LAB = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "datasets").is_dir() and (p / "labkit").is_dir())   # lab root (works from any folder)
sys.path.insert(0, str(LAB))                                # make `labkit` importable from the notebook
from labkit.mock_api import MockSession, BASE_URL
import requests

session = MockSession()                                     # offline fake with the same surface as requests.Session
resp = session.get(f"{BASE_URL}/customers", params={"page": 1, "page_size": 3}, timeout=5)
print(resp.status_code, resp.json()["total"], [c["customer_id"] for c in resp.json()["data"]])
# Prefer real HTTP? In a terminal:  python -m labkit.mock_server --port 8765
# then use requests.Session() with base_url "http://127.0.0.1:8765/v1" — your functions shouldn't change at all.

## Exercise 1 — 🟢 Easy · Classify retryable failures

**Implement:** `is_retryable(outcome) -> bool`

`outcome` is either an HTTP status code (int) or an exception instance. Retryable: 408, 425, 429, 500, 502, 503, 504, `requests.Timeout`, `requests.ConnectionError`, built-in `TimeoutError`/`ConnectionError`. Everything else (incl. other 4xx, `ValueError`) is not.

**Input:** `outcome`

**Expected output:** `bool`

**Constraints:** Standard library + `requests`-compatible session objects; always pass `timeout=`.

**Example:** the tests are executable examples — read `test_is_retryable` in `tests/apis/test_07_04_retries_rate_limits.py`.

**Check:** `pytest tests/apis/test_07_04_retries_rate_limits.py -k test_is_retryable`

**YOUR CODE ↓**

## Exercise 2 — 🟡 Medium · Backoff schedule with optional jitter

**Implement:** `backoff_delays(retries: int, base: float = 0.5, factor: float = 2.0, max_delay: float = 30.0, jitter=None) -> list[float]`

Return the delays before each retry: `base * factor**i`, capped at `max_delay`. If `jitter` is a function, apply it to each capped delay (e.g. `lambda d: random.uniform(0, d)` — 'full jitter'). Round to 3 decimals.

**Input:** `retries: int, base: float = 0.5, factor: float = 2.0, max_delay: float = 30.0, jitter=None`

**Expected output:** `list[float]`

**Constraints:** Standard library + `requests`-compatible session objects; always pass `timeout=`.

**Example**

```python
backoff_delays(5)
# -> [0.5, 1.0, 2.0, 4.0, 8.0]
```

**Check:** `pytest tests/apis/test_07_04_retries_rate_limits.py -k test_backoff_delays`

**YOUR CODE ↓**

## Exercise 3 — 🟡 Medium · Parse Retry-After

**Implement:** `parse_retry_after(value, now: datetime | None = None) -> float | None`

`Retry-After` is either delta-seconds (`"120"`) or an HTTP date (`"Wed, 21 Oct 2026 07:28:00 GMT"`). Return seconds to wait (≥ 0, past dates → 0.0), or `None` for missing/invalid. `now` defaults to current UTC time (inject it in tests). Use `email.utils.parsedate_to_datetime`.

**Input:** `value, now: datetime | None = None`

**Expected output:** `float | None`

**Constraints:** Standard library + `requests`-compatible session objects; always pass `timeout=`.

**Example**

```python
parse_retry_after('120')
# -> 120.0
```

**Check:** `pytest tests/apis/test_07_04_retries_rate_limits.py -k test_parse_retry_after`

**YOUR CODE ↓**

## Exercise 4 — 🟠 Hard · GET with retries, backoff and Retry-After

**Implement:** `get_with_retry(session, url: str, params=None, max_attempts: int = 4, base_delay: float = 0.5, sleep=time.sleep, timeout: float = 10)`

Call `session.get(url, params=params, timeout=timeout)`. On retryable statuses or exceptions, wait and try again up to `max_attempts` attempts total.
Wait = `Retry-After` header seconds when present on a 429/503, else `base_delay * 2**(attempt-1)`. After the last attempt: for an exception re-raise it; for a status call
`raise_for_status()`. Non-retryable error statuses raise immediately via `raise_for_status()`. Return the successful response. Attach `get_with_retry.last_attempts`.

**Input:** `session, url: str, params=None, max_attempts: int = 4, base_delay: float = 0.5, sleep=time.sleep, timeout: float = 10`

**Constraints:** Standard library + `requests`-compatible session objects; always pass `timeout=`.

**Example:** the tests are executable examples — read `test_get_with_retry_recovers, test_get_with_retry_gives_up, test_get_with_retry_non_retryable_and_exception` in `tests/apis/test_07_04_retries_rate_limits.py`.

**Check:** `pytest tests/apis/test_07_04_retries_rate_limits.py -k "test_get_with_retry_recovers or test_get_with_retry_gives_up or test_get_with_retry_non_retryable_and_exception"`

**YOUR CODE ↓**

## Challenge — 🔴 Very Hard · Resilient client: token bucket + retries + circuit breaker · 🧩 Mini-project

**Implement:** `class CircuitOpen(Exception)`, `class ResilientClient`

`ResilientClient(session, base_url, rate_per_sec=10, burst=10, max_attempts=3, failure_threshold=5, cooldown=30, clock=time.monotonic, sleep=time.sleep)`
with `get(path, **params) -> dict` (parsed JSON) and a `metrics` dict (`requests`, `retries`, `throttled_waits`, `circuit_rejections`).

1. **Token bucket**: capacity `burst`, refilled at `rate_per_sec`. Each HTTP attempt consumes a token; if none is available, `sleep` exactly long enough for one token (count a throttled wait).
2. **Retries**: retry retryable statuses/exceptions with `0.5 * 2**(n-1)` backoff (or `Retry-After`) up to `max_attempts`; non-retryable 4xx → `raise_for_status()`.
3. **Circuit breaker**: count *consecutive failed calls* (a call fails after its retries are exhausted). At `failure_threshold` the circuit opens: further `get`s raise `CircuitOpen`
   immediately without touching the session, until `cooldown` seconds pass; then one trial call is allowed (half-open): success closes the circuit, failure re-opens it.
Use the fake clock and sleep from the tests to reason about time: the fake `sleep` should advance the fake clock.

**Input:** see the class description above

**Expected output:** an instance behaving as described

**Constraints:** Standard library + `requests`-compatible session objects; always pass `timeout=`.

**Example:** the tests are executable examples — read `test_token_bucket_throttles, test_retries_then_success, test_circuit_breaker_opens_and_recovers` in `tests/apis/test_07_04_retries_rate_limits.py`.

**Check:** `pytest tests/apis/test_07_04_retries_rate_limits.py -k "test_token_bucket_throttles or test_retries_then_success or test_circuit_breaker_opens_and_recovers or test_404_does_not_trip_breaker"`

**YOUR CODE ↓**